In [ ]:
import numpy as np
import psfsim
import matplotlib.pyplot as plt
import matplotlib as mpl

from psfsim.polychrom import PolychromaticPSF

In [ ]:
z087 = np.linspace(0.76, 0.977, 10)
k213 = np.linspace(1.95, 2.3, 10)
w146 = np.linspace(0.927, 2.0, 10)
j129 = np.linspace(1.131, 1.454, 10)

ps_size = 96
ovsamp = pixel_size = 8
mag = 1e9  # some unit
step = 0.1  # degrees
borders = (0.2, 1.0)

bound = ps_size // 2
center = np.array((ps_size // 2, ps_size // 2))
flux_factor = pixel_size**2 * mag

In [ ]:
def line_coords(angle, bound, center=np.array((0, 0))):
    # since (0, 0) in data coords is top left instead of bottom left we must use clockwise rotations to get
    # a visually ccl rotation. However we will take a transpose later anyways so the rotation matrix def is ccl, as expected. How quaint
    angle = np.deg2rad(angle)
    # print( angle )
    line = np.zeros((2, bound))
    line[0] = np.arange(bound)

    rotation = np.array(
        (
            (np.cos(angle), -np.sin(angle)),
            (np.sin(angle), np.cos(angle)),
        )
    )

    if len(rotation.shape) > 2:
        line = line.reshape((1, 2, bound))

    # print( line.shape )
    # print( rotation.T.shape )

    return np.int32(rotation.T @ line) + np.int32(np.reshape(center, (2, 1)))


def find_spikes(
    image, step, bound=1024, center=np.array((0, 0)), borders=(0.0, 1.0), threshold=0.6, verbose=False
):
    angles = np.linspace(0, 360, num=np.int32(360.0 / step), endpoint=False)

    lines = line_coords(angles, bound, center)
    image_analyze = (
        image.T
    )  # imshow will display the image as row,col with 0,0 in top left. For our analysis we want (col,row).

    line_vals = image_analyze[lines[:, 0], lines[:, 1]]
    sums = np.sum(line_vals, axis=-1)

    # 0 to 1 for these. Should I add a way to make it force 0 <= borders[0] < borders[1] <= 1?
    range_low = np.int_(borders[0] * bound)
    range_high = np.int_(borders[1] * bound)

    sums = np.sum(line_vals[:, range_low:range_high], axis=-1)
    """
    This formula is slightly different from the midpoint formula from before. The formula defaults to the 60% value, using 50% recovers the original behavior. Other
    ways to do this include rejecting the lowest 10% of values or so. That one might be worth pursuing in the future. The goal is to keep this method as simple as possible--
    I do NOT want to start staring at noise to find spikes.

    There is another method that may be worth pursuing that involves a boxcar average that goes around the circle and might be able to correct for groups of spikes that are much
    higher or lower than the naive cutoff assumes.
    """
    diff = np.max(sums) - np.min(sums)
    cutoff = np.min(sums) + (diff * threshold)
    spike_indices = np.where(sums > cutoff)[0]
    spike_angles = angles[spike_indices]
    # print( np.diff( spike_angles ) )

    # try 3 degrees to tell differing spikes apart for now
    spike_angle_discriminator = 3.0
    borders = np.where(np.diff(spike_angles) > spike_angle_discriminator)[0]
    spike_groups = np.split(spike_angles, borders + 1)
    # print( spike_groups )

    # unfortunately this for loop is necessary unless theres a convenient package for jagged arrays
    spike_list = np.zeros(len(spike_groups))
    for i in np.arange(len(spike_groups)):
        spike_list[i] = np.median(spike_groups[i])

    # print( spike_list )
    if verbose:
        return sums, spike_angles, cutoff, spike_list
    else:
        return spike_list


def draw_ray(ax, angle, bound, center=np.array((0, 0)), borders=(0.0, 1.0), **kwargs):
    line = line_coords(angle, bound, center)
    range_low = np.int_(borders[0] * bound)
    range_high = np.int_(borders[1] * bound)
    ax.plot(line[0, range_low:range_high], line[1, range_low:range_high], **kwargs)
    return


def downsample_2d_image(image, pixel_size=8):  # ONLY WORKS ON 2D IMAGES
    subpixel_side = np.arange(pixel_size)
    subpixel_cols, subpixel_rows = np.meshgrid(subpixel_side, subpixel_side)

    dsamp_size = image.shape[0] // pixel_size
    pixels_side = np.arange(dsamp_size) * pixel_size
    pixels_cols, pixels_rows = np.meshgrid(pixels_side, pixels_side)

    pixels_rows = pixels_rows[:, :, np.newaxis]  # promote to 3D
    pixels_cols = pixels_cols[:, :, np.newaxis]

    idx_arr_subpixel_rows = np.full((dsamp_size, dsamp_size, pixel_size**2), subpixel_rows.flatten())
    idx_arr_subpixel_cols = np.full((dsamp_size, dsamp_size, pixel_size**2), subpixel_cols.flatten())

    idx_arr_rows = idx_arr_subpixel_rows + pixels_rows
    idx_arr_cols = idx_arr_subpixel_cols + pixels_cols

    pixelated_image = image[idx_arr_rows, idx_arr_cols]
    downsampled_image = np.mean(pixelated_image, axis=-1)
    return downsampled_image


def poisson_resample_image(img, flux_factor):
    # resamples an image where each pixel gets its own poisson distribution
    rng = np.random.default_rng()
    poisson_means = img * flux_factor

    resampled_img = rng.poisson(poisson_means, size=poisson_means.shape) / flux_factor
    return resampled_img

In [ ]:
z087_obj = PolychromaticPSF(9, 0, 0, z087)
k213_obj = PolychromaticPSF(9, 0, 0, k213)
w146_obj = PolychromaticPSF(9, 0, 0, w146)
j129_obj = PolychromaticPSF(9, 0, 0, j129)

z087_obj.compute_poly_psf(
    optical_psf_only=True, use_postage_stamp_size=ps_size, ovsamp=ovsamp, use_filter="Z087"
)
z087_psf_ideal = np.log10(
    poisson_resample_image(downsample_2d_image(np.abs(z087_obj.chromatic_psf), pixel_size), flux_factor)
)

w146_obj.compute_poly_psf(
    optical_psf_only=True, use_postage_stamp_size=ps_size, ovsamp=ovsamp, use_filter="W146"
)
w146_psf_ideal = np.log10(
    poisson_resample_image(downsample_2d_image(np.abs(w146_obj.chromatic_psf), pixel_size), flux_factor)
)

k213_obj.compute_poly_psf(
    optical_psf_only=True, use_postage_stamp_size=ps_size, ovsamp=ovsamp, use_filter="K213"
)
k213_psf_ideal = np.log10(
    poisson_resample_image(downsample_2d_image(np.abs(k213_obj.chromatic_psf), pixel_size), flux_factor)
)

j129_obj.compute_poly_psf(
    optical_psf_only=True, use_postage_stamp_size=ps_size, ovsamp=ovsamp, use_filter="J129"
)
j129_psf_ideal = np.log10(
    poisson_resample_image(downsample_2d_image(np.abs(j129_obj.chromatic_psf), pixel_size), flux_factor)
)

In [ ]:
spikes_z087_ideal = find_spikes(z087_psf_ideal, step, bound, center, borders=borders)
sums, groups_k213_ideal, cutoff, spikes_k213_ideal = find_spikes(
    k213_psf_ideal, step, bound, center, borders=borders, verbose=True
)
spikes_w146_ideal = find_spikes(w146_psf_ideal, step, bound, center, borders=borders)
spikes_j129_ideal = find_spikes(j129_psf_ideal, step, bound, center, borders=borders)


fig, ax = plt.subplots(figsize=(12, 18), nrows=3, ncols=2)

im = ax[0, 0].imshow(z087_psf_ideal)
fig.colorbar(im, ax=ax[0, 0])

im = ax[0, 1].imshow(k213_psf_ideal)
fig.colorbar(im, ax=ax[0, 1])

im = ax[1, 0].imshow(w146_psf_ideal)
fig.colorbar(im, ax=ax[1, 0])

im = ax[1, 1].imshow(j129_psf_ideal)
fig.colorbar(im, ax=ax[1, 1])


# for s in spikes_k213_ideal:
#     draw_ray( ax[0,1], s, bound, center, borders=borders, color='C3' )

# for s00, s10, s11 in zip(
#     spikes_z087_ideal,
#     spikes_w146_ideal,
#     spikes_j129_ideal
# ):
#     draw_ray( ax[0,0], s00, bound, center, borders=borders, color='C3' )
#     draw_ray( ax[1,0], s10, bound, center, borders=borders, color='C3' )
#     draw_ray( ax[1,1], s11, bound, center, borders=borders, color='C3' )

ax[2, 0].plot(sums)
ax[2, 0].axhline(cutoff, ls="--", color="C3")

# ax[0,1].set_xlim( 25, 41 )
# ax[0,1].set_ylim( 18, 2 )

In [ ]:
rng = np.random.default_rng()
test_aberrations = rng.random(5) * 0.1
# test_aberrations = ( None, None, 0.2 )
print(f"Test Aberration Values: {test_aberrations}")

k213_obj.compute_poly_psf(
    optical_psf_only=True,
    use_postage_stamp_size=ps_size,
    ovsamp=ovsamp,
    use_filter="K213",
    extra_aberrations=test_aberrations,
)
k213_psf_aberrations = np.log10(
    poisson_resample_image(downsample_2d_image(np.abs(k213_obj.chromatic_psf), pixel_size), flux_factor)
)

In [ ]:
print(f"Test Aberration Values: {test_aberrations}")

test_borders = (0.3, 1.0)

fig2, ax2 = plt.subplots(figsize=(18, 6), ncols=3)
sums, _, cutoff, spikes_k213_aberrations = find_spikes(
    k213_psf_aberrations, step, bound, center, borders=test_borders, verbose=True
)

im = ax2[0].imshow(k213_psf_aberrations)
fig.colorbar(im, ax=ax2[0])

for s in spikes_k213_aberrations:
    draw_ray(ax2[0], s, bound, center, borders=test_borders, color="C3")

N = np.arange(spikes_k213_ideal.size)
ax2[1].scatter(N, spikes_k213_aberrations - spikes_k213_ideal)
ax2[1].axhline(0, color="C3", ls="--")

ax2[2].plot(sums)
ax2[2].axhline(cutoff, color="C3", ls="--")

sorted_sums = np.sort(sums)
low_idx = 15 * sorted_sums.size // 100
cut_sums = sorted_sums[low_idx:]
diff = np.max(cut_sums) - np.min(cut_sums)
mod_cutoff = np.min(cut_sums) + (diff * 0.6)
ax2[2].axhline(mod_cutoff, color="C4", ls="--")
ax2[2].axhline(cut_sums[0], color="C5", ls="--")
# ax2[2].set_xlim( 1000, 1500 )

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6), ncols=2)

im = ax[0].imshow(z087_psf_ideal, origin="lower")
fig.colorbar(im, ax=ax[0])

im = ax[1].imshow(z087_psf_ideal, origin="lower")
fig.colorbar(im, ax=ax[1])

for s in spikes_z087_ideal:
    draw_ray(ax[1], s, bound, center, borders=borders, color="C3")

ax[0].set_xlabel("X Pixel")
ax[0].set_ylabel("Y Pixel")
ax[0].set_title("Ideal Z087 PSF")

ax[1].set_xlabel("X Pixel")
ax[1].set_ylabel("Y Pixel")
ax[1].set_title("Ideal Z087 PSF + Found Spikes")

In [ ]:
z087_obj.compute_poly_psf(
    optical_psf_only=True,
    use_postage_stamp_size=ps_size,
    ovsamp=ovsamp,
    use_filter="Z087",
    extra_aberrations=(5.0,),
)
z087_psf_hcenter = np.log10(
    poisson_resample_image(downsample_2d_image(np.abs(z087_obj.chromatic_psf), pixel_size), flux_factor)
)

In [ ]:
z087_obj.compute_poly_psf(
    optical_psf_only=True,
    use_postage_stamp_size=ps_size,
    ovsamp=ovsamp,
    use_filter="Z087",
    extra_aberrations=(
        None,
        5.0,
    ),
)
z087_psf_vcenter = np.log10(
    poisson_resample_image(downsample_2d_image(np.abs(z087_obj.chromatic_psf), pixel_size), flux_factor)
)

In [ ]:
z087_obj.compute_poly_psf(
    optical_psf_only=True,
    use_postage_stamp_size=ps_size,
    ovsamp=ovsamp,
    use_filter="Z087",
    extra_aberrations=(
        None,
        None,
        1.0,
    ),
)
z087_psf_focus = np.log10(
    poisson_resample_image(downsample_2d_image(np.abs(z087_obj.chromatic_psf), pixel_size), flux_factor)
)

In [ ]:
z087_obj.compute_poly_psf(
    optical_psf_only=True,
    use_postage_stamp_size=ps_size,
    ovsamp=ovsamp,
    use_filter="Z087",
    extra_aberrations=(
        None,
        None,
        None,
        1.0,
    ),
)
z087_psf_z5 = np.log10(
    poisson_resample_image(downsample_2d_image(np.abs(z087_obj.chromatic_psf), pixel_size), flux_factor)
)

In [ ]:
z087_obj.compute_poly_psf(
    optical_psf_only=True,
    use_postage_stamp_size=ps_size,
    ovsamp=ovsamp,
    use_filter="Z087",
    extra_aberrations=(
        None,
        None,
        None,
        None,
        1.0,
    ),
)
z087_psf_z6 = np.log10(
    poisson_resample_image(downsample_2d_image(np.abs(z087_obj.chromatic_psf), pixel_size), flux_factor)
)

In [ ]:
mpl.rcParams["font.size"] = 15

fig, ax = plt.subplots(figsize=(21, 12), nrows=2, ncols=3)

im = ax[0, 0].imshow(z087_psf_ideal)
ax[0, 0].set_title("Ideal Z087 PSF")

im = ax[0, 1].imshow(z087_psf_hcenter)
ax[0, 1].set_title("Horizontally Off-Center")

im = ax[0, 2].imshow(z087_psf_vcenter)
ax[0, 2].set_title("Vertically Off-Center")

im = ax[1, 0].imshow(z087_psf_focus)
ax[1, 0].set_title("Unfocused")

im = ax[1, 1].imshow(z087_psf_z5)
ax[1, 1].set_title("Z5 Astigmatism (X Shape)")

im = ax[1, 2].imshow(z087_psf_z6)
ax[1, 2].set_title("Z6 Astigmatism (+ Shape)")

for a in ax.flatten():
    a.set_xlabel("X Pixel")
    a.set_ylabel("Y Pixel")

fig.subplots_adjust(right=0.8)
cbar = fig.add_axes([0.83, 0.15, 0.025, 0.7])
fig.colorbar(im, cax=cbar)

In [ ]:
z087_obj.compute_poly_psf(
    optical_psf_only=True,
    use_postage_stamp_size=ps_size,
    ovsamp=ovsamp,
    use_filter="Z087",
    extra_aberrations=(
        None,
        0.1,
    ),
)
z087_psf_vcenter_small = np.log10(
    poisson_resample_image(downsample_2d_image(np.abs(z087_obj.chromatic_psf), pixel_size), flux_factor)
)

z087_obj.compute_poly_psf(
    optical_psf_only=True,
    use_postage_stamp_size=ps_size,
    ovsamp=ovsamp,
    use_filter="Z087",
    extra_aberrations=(
        None,
        -0.1,
    ),
)
z087_psf_vcenter_neg = np.log10(
    poisson_resample_image(downsample_2d_image(np.abs(z087_obj.chromatic_psf), pixel_size), flux_factor)
)

In [ ]:
z087_obj.compute_poly_psf(
    optical_psf_only=True,
    use_postage_stamp_size=ps_size,
    ovsamp=ovsamp,
    use_filter="Z087",
    extra_aberrations=(0.1,),
)
z087_psf_hcenter_small = np.log10(
    poisson_resample_image(downsample_2d_image(np.abs(z087_obj.chromatic_psf), pixel_size), flux_factor)
)

z087_obj.compute_poly_psf(
    optical_psf_only=True,
    use_postage_stamp_size=ps_size,
    ovsamp=ovsamp,
    use_filter="Z087",
    extra_aberrations=(-0.1,),
)
z087_psf_hcenter_neg = np.log10(
    poisson_resample_image(downsample_2d_image(np.abs(z087_obj.chromatic_psf), pixel_size), flux_factor)
)

In [ ]:
z087_obj.compute_poly_psf(
    optical_psf_only=True,
    use_postage_stamp_size=ps_size,
    ovsamp=ovsamp,
    use_filter="Z087",
    extra_aberrations=(
        None,
        None,
        0.1,
    ),
)
z087_psf_focus_small = np.log10(
    poisson_resample_image(downsample_2d_image(np.abs(z087_obj.chromatic_psf), pixel_size), flux_factor)
)

z087_obj.compute_poly_psf(
    optical_psf_only=True,
    use_postage_stamp_size=ps_size,
    ovsamp=ovsamp,
    use_filter="Z087",
    extra_aberrations=(
        None,
        None,
        -0.1,
    ),
)
z087_psf_focus_neg = np.log10(
    poisson_resample_image(downsample_2d_image(np.abs(z087_obj.chromatic_psf), pixel_size), flux_factor)
)

In [ ]:
spikes_z087_vcenter_small = find_spikes(z087_psf_vcenter_small, step, bound, center, borders=borders)
spikes_z087_vcenter_neg = find_spikes(z087_psf_vcenter_neg, step, bound, center, borders=borders)
spikes_z087_hcenter_small = find_spikes(z087_psf_hcenter_small, step, bound, center, borders=borders)
spikes_z087_hcenter_neg = find_spikes(z087_psf_hcenter_neg, step, bound, center, borders=borders)
spikes_z087_focus_small = find_spikes(z087_psf_focus_small, step, bound, center, borders=borders)
spikes_z087_focus_neg = find_spikes(z087_psf_focus_neg, step, bound, center, borders=borders)
N = np.arange(spikes_z087_focus_small.size)

fig, ax = plt.subplots(figsize=(14, 6), ncols=2)

ax[0].scatter(N, spikes_z087_vcenter_small - spikes_z087_ideal, label="Positive (Down)")
ax[0].scatter(N, spikes_z087_vcenter_neg - spikes_z087_ideal, label="Negative (Up)")
ax[0].axhline(0, color="C3", ls="--")
ax[0].set_title("Vertical Centering")
ax[0].set_xlabel("Spike Number")
ax[0].set_ylabel("Angle Difference from Ideal (degrees)")
ax[0].set_xticks(N)
ax[0].legend()

# ax[0].scatter( N, spikes_z087_hcenter_small - spikes_z087_ideal )
# ax[0].scatter( N, spikes_z087_hcenter_neg - spikes_z087_ideal )
# ax[0].axhline( 0, color='C3', ls='--' )

ax[1].scatter(N, spikes_z087_focus_small - spikes_z087_ideal, label="Positive")
ax[1].scatter(N, spikes_z087_focus_neg - spikes_z087_ideal, label="Negative")
ax[1].axhline(0, color="C3", ls="--")
ax[1].set_title("Focusing")
ax[1].set_xlabel("Spike Number")
ax[1].set_ylabel("Angle Difference from Ideal (degrees)")
ax[1].set_xticks(N)
ax[1].legend()

fig.suptitle("Response of Z087 PSF to 0.1 Amplitude Aberrations")